# 07 — Streaming Simulation

**Week:** 10

Goal: Simulate new JSON events using Auto Loader / Structured Streaming and write a Streaming Bronze table.


In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, TimestampType

event_schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("event_timestamp", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("entity_id", StringType(), True),
    StructField("severity", StringType(), True),
])

stream_input_path = "/Volumes/workspace/default/<project_name>/streaming_input/"
checkpoint_path = "/Volumes/workspace/default/<project_name>/checkpoints/streaming_bronze_events/"

stream_df = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .schema(event_schema)
        .load(stream_input_path)
)

display(stream_df)


In [ ]:
query = (
    stream_df.writeStream
        .format("delta")
        .option("checkpointLocation", checkpoint_path)
        .outputMode("append")
        .table("bronze_streaming_events")
)

# In Databricks, keep the stream running while adding new JSON event files.
print("Streaming query started.")


In [ ]:
%sql
SELECT event_type, severity, COUNT(*) AS event_count
FROM bronze_streaming_events
GROUP BY event_type, severity
ORDER BY event_count DESC


## Kafka Design Note

Kafka implementation is not mandatory. Document Kafka-style event design in `streaming/kafka_event_schema.json` and `streaming/structured_streaming_design.md`.
